# Notebook 09 — Predictive Entropy Uncertainty Estimation
**Project**: Uncertainty-Aware Terrain Semantic Segmentation and Traversability Estimation for Autonomous Planetary Rover Navigation  
**Stage A**: Predictive Entropy Uncertainty Analysis on Verified Epoch 24 Multimodal Model

---

## 1. Research Objectives
This notebook evaluates the hypothesis:
> *"Does predictive uncertainty identify pixels where the multimodal segmentation model is more likely to be incorrect?"*

We employ **Shannon Predictive Entropy** derived from class posterior probabilities:
$$H(p) = -\sum_{c=0}^{C-1} p_c \ln(p_c)$$
Normalized to $[0, 1]$ via the maximum theoretical entropy $\ln(C) = \ln(8) \approx 2.0794$:
$$H_{\text{norm}}(p) = \frac{H(p)}{\ln(8)} \in [0, 1]$$

Where:
- $H_{\text{norm}}(p) = 0.0$: Maximum confidence (model assigns probability 1.0 to a single terrain class).
- $H_{\text{norm}}(p) = 1.0$: Complete uncertainty (uniform distribution over all 8 terrain classes).

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else (next((p for p in (cwd, *cwd.parents) if (p / 'notebooks').is_dir()), cwd))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "uncertainty"

# Verify summary metrics
metrics_path = OUTPUT_DIR / "uncertainty_metrics.json"
with open(metrics_path, "r") as f:
    metrics = json.load(f)

print("=== Predictive Entropy Uncertainty Metrics ===")
print(f"Validation Frames Evaluated: {metrics['validation_samples']}")
print(f"Total Validation Pixels:    {metrics['total_pixels']:,}")
print(f"Mean Entropy (Correct):     {metrics['overall_uncertainty']['mean_entropy_correct']:.4f}")
print(f"Mean Entropy (Incorrect):   {metrics['overall_uncertainty']['mean_entropy_incorrect']:.4f}")
print(f"Entropy Separation Ratio:   {metrics['overall_uncertainty']['entropy_separation_ratio']:.2f}x")
print(f"AUROC (Error Detection):    {metrics['overall_uncertainty']['auroc_error_detection']:.4f}")

## 2. Uncertainty Bins vs. Empirical Error Rate
Pixels are partitioned into 5 uncertainty intervals $[0.0, 0.2), [0.2, 0.4), [0.4, 0.6), [0.6, 0.8), [0.8, 1.0]$.
A reliable uncertainty metric should exhibit a strong monotonic relationship between uncertainty and segmentation error rate.

In [ ]:
df_bins = pd.read_csv(OUTPUT_DIR / "uncertainty_error_analysis.csv")
print(df_bins.to_markdown(index=False))

# Plot Error Rate vs Uncertainty Bin
fig, ax1 = plt.subplots(figsize=(8, 5))
color = 'tab:red'
ax1.set_xlabel('Predictive Entropy Bin', fontsize=12, fontweight='bold')
ax1.set_ylabel('Error Rate (%)', color=color, fontsize=12, fontweight='bold')
bars = ax1.bar(df_bins['bin_range'], df_bins['error_rate_pct'], color=color, alpha=0.7, edgecolor='black', width=0.5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 100)

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')

plt.title('Predictive Uncertainty vs. Segmentation Error Rate', fontsize=14, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Per-Class Uncertainty Analysis
Examines whether difficult terrain classes (such as Rock, Bedrock, and Grass) manifest higher predictive entropy compared to expansive homogeneous terrain (Compact, Sandy).

In [ ]:
df_class = pd.read_csv(OUTPUT_DIR / "entropy_summary.csv")
print(df_class.to_markdown(index=False))

## 4. Qualitative Visualizations: Uncertainty & Error Maps
Each sample displays:
1. Input RGB
2. Ground-Truth Mask
3. Predicted Semantic Mask
4. Predictive Entropy Map (Normalized Shannon Entropy)
5. Prediction Error Map (Red = Segmentation Error)

In [ ]:
pred_imgs = sorted((OUTPUT_DIR / "predictions").glob("*.png"))
for p in pred_imgs:
    print(f"Displaying: {p.name}")
    img = Image.open(p)
    plt.figure(figsize=(16, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()